# SI Figure S6: per-solvent test RMSE of six solvent-correction models

Per nucleus, the per-solvent test RMSE of six solvent-correction models: SotA implicit (single-slope
PCM), the paper's implicit fit (stationary + PCM as separate terms), and explicit (Desmond), each
with and without a vibrational correction (QCD for ¹H, Desmond vibration for ¹³C). Level of theory:
dsd_pbep86/pcSseg3 on pbe0_tz geometries, PCM substituted from b3lyp_d3bj/pcSseg3.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt

import delta22
import delta22_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
query = delta22.add_composite_columns(delta22.load_query_df_dft(
    DELTA22_HDF5, XLSX, pcm_reference_method="b3lyp_d3bj", pcm_reference_basis="pcSseg3", verbose=False))
solutes = sorted(query["solute"].unique())

METHOD, BASIS, GEOM = "dsd_pbep86", "pcSseg3", "pbe0_tz"

# same six categories for both nuclei; the "+ Vibrations" term differs (qcd for H, desmond_vib for C)
LADDER = {
    "H": {
        "stationary_plus_pcm": "SotA Implicit Solvent",
        "stationary_plus_pcm_plus_qcd": "SotA Implicit Solvent + Vibrations",
        "stationary + pcm": "Implicit Solvent",
        "stationary_plus_qcd + pcm": "Implicit Solvent + Vibrations",
        "stationary + desmond": "Explicit Solvent",
        "stationary_plus_qcd + desmond": "Explicit Solvent + Vibrations"},
    "C": {
        "stationary_plus_pcm": "SotA Implicit Solvent",
        "stationary_plus_pcm_plus_des_vib": "SotA Implicit Solvent + Vibrations",
        "stationary + pcm": "Implicit Solvent",
        "stationary_plus_des_vib + pcm": "Implicit Solvent + Vibrations",
        "stationary + desmond": "Explicit Solvent",
        "stationary_plus_des_vib + desmond": "Explicit Solvent + Vibrations"},
}

In [ ]:
# one dark/light pair per solvent-treatment family (SotA implicit / implicit / explicit), light = +Vibrations
S6_COLORS = ["#C4B037", "#F5EDA0", "#A72608", "#E4A0A0", "#5F7C8A", "#B9E7DF"]

## Formula ladder per nucleus

In [ ]:
# the published panels use 250 seeded train/test splits
N_SPLITS = 250

In [ ]:
for nucleus, labels in LADDER.items():
    results = delta22.fig3d_formula_regressions(query, METHOD, BASIS, GEOM, list(labels),
                                                delta22.DESMOND_SOLVENTS, n_splits=N_SPLITS,
                                                solutes=solutes, nucleus=nucleus)
    med = results.groupby("formula")["test_RMSE"].median()
    print(f"--- {nucleus} ({METHOD}) median test RMSE ---")
    for formula in labels:
        print(f"  {labels[formula]:38s} {med[formula]:.4f}")
    delta22_plots.plot_formula_ladder_boxplot(
        results, labels, delta22.SOLVENT_GROUPS, nucleus=nucleus, colors=S6_COLORS,
        title=f"Explicit Solvation + Vibrational Effects Improve Predictions Across Solvents ({nucleus} Nucleus)",
        save_path=figure_path(f"si_figure_s06_ladder_{'1H' if nucleus == 'H' else '13C'}.png"))
plt.show()